In [1]:
import torch


# Create the DML device instance
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Move your model and data to the DML device


Using device: cuda


In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split


In [3]:
# DEVICE (DirectML)
# -------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


In [4]:
# -------------------------------------------------
# LOAD YOUR DATA
# Replace 'your_ids.csv' and label column name
# -------------------------------------------------
df = pd.read_csv("combined_dataset.csv")
df=df.drop(columns=df.columns[0])  # drop non-numeric columns

df.columns = df.columns.str.strip()
df['Label'] = df['Label'].str.strip()
print(df['Label'].unique())
df.replace([np.inf, -np.inf], np.nan, inplace=True)






['BENIGN' 'DDoS' 'PortScan' 'Bot' 'Infiltration'
 'Web Attack � Brute Force' 'Web Attack � XSS'
 'Web Attack � Sql Injection' 'FTP-Patator' 'SSH-Patator' 'DoS slowloris'
 'DoS Slowhttptest' 'DoS Hulk' 'DoS GoldenEye' 'Heartbleed']


In [5]:
df=df.dropna()  # Drop rows with NaN values

In [6]:
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

# Separate
X = df.drop("Label", axis=1).values
y = df["Label"].values




In [7]:
from sklearn.model_selection import train_test_split

# X = features, y = labels (0 = normal, 1 = fail)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Validation split (CRITICAL)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_norm = scaler.fit_transform(X_train)
X_val_norm   = scaler.transform(X_val)
X_test_norm  = scaler.transform(X_test)

In [9]:
X_train_normal = X_train_norm[y_train == 0]
X_val_normal   = X_val_norm[y_val == 0]

In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32).to(device)
X_val_tensor   = torch.tensor(X_val_norm, dtype=torch.float32).to(device)
X_test_tensor  = torch.tensor(X_test_norm, dtype=torch.float32).to(device)

X_train_normal_tensor = torch.tensor(X_train_normal, dtype=torch.float32).to(device)
X_val_normal_tensor   = torch.tensor(X_val_normal, dtype=torch.float32).to(device)

In [11]:
input_dim = X_train_normal.shape[1]
print("Input dimension:", input_dim)

Input dimension: 77


In [12]:
class DAE(nn.Module):
    def __init__(self, input_dim):
        super(DAE, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32)

        )

        self.decoder = nn.Sequential(

            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

model = DAE(input_dim).to(device)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.0001,momentum=0.9)

# -------------------------------------------------
# TRAINING
# -------------------------------------------------







In [13]:
epochs = 60
batch_size = 512
noise_factor = 0.12

In [14]:


def train_epoch(X_data):
    model.train()
    perm = torch.randperm(X_data.size(0))
    total_loss = 0

    for i in range(0, X_data.size(0), batch_size):
        idx = perm[i:i+batch_size]
        batch = X_data[idx]

        noise = noise_factor * torch.randn_like(batch)
        noisy_batch = batch + noise

        optimizer.zero_grad()
        outputs = model(noisy_batch)
        loss = criterion(outputs, batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss



In [15]:
criterion = torch.nn.MSELoss()

best_val_loss = float('inf')
patience = 15
counter = 0

for epoch in range(epochs):  # no need 100

    # -------- TRAIN --------
    model.train()
    optimizer.zero_grad()

    noisy = X_train_normal_tensor + noise_factor * torch.randn_like(X_train_normal_tensor)
    recon, _ = model(noisy)
    train_loss = criterion(recon, X_train_normal_tensor)
    train_loss.backward()
    optimizer.step()

    # -------- VALIDATION --------
    model.eval()
    with torch.no_grad():
        noisy_val = X_val_normal_tensor + noise_factor * torch.randn_like(X_val_normal_tensor)
        val_output, _ = model(noisy_val)

        val_loss = criterion(val_output, X_val_normal_tensor)

    print(f"Epoch {epoch+1}, Train Loss: {train_loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

    # -------- EARLY STOPPING --------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

Epoch 1, Train Loss: 0.8054, Val Loss: 1.7477
Epoch 2, Train Loss: 0.8054, Val Loss: 1.7478
Epoch 3, Train Loss: 0.8054, Val Loss: 1.7479
Epoch 4, Train Loss: 0.8053, Val Loss: 1.7480
Epoch 5, Train Loss: 0.8053, Val Loss: 1.7481
Epoch 6, Train Loss: 0.8053, Val Loss: 1.7482
Epoch 7, Train Loss: 0.8053, Val Loss: 1.7483
Epoch 8, Train Loss: 0.8053, Val Loss: 1.7484
Epoch 9, Train Loss: 0.8053, Val Loss: 1.7485
Epoch 10, Train Loss: 0.8053, Val Loss: 1.7486
Epoch 11, Train Loss: 0.8053, Val Loss: 1.7487
Epoch 12, Train Loss: 0.8053, Val Loss: 1.7488
Epoch 13, Train Loss: 0.8053, Val Loss: 1.7489
Epoch 14, Train Loss: 0.8053, Val Loss: 1.7490
Epoch 15, Train Loss: 0.8053, Val Loss: 1.7491
Epoch 16, Train Loss: 0.8053, Val Loss: 1.7491
Early stopping triggered


In [16]:
def extract_features(model, X):
    model.eval()

    X_tensor = torch.from_numpy(X).float().to(device)

    with torch.no_grad():
        recon, latent = model(X_tensor)
        error = torch.mean((X_tensor - recon) ** 2, dim=1)

    return (
        error.cpu().numpy().reshape(-1, 1),
        latent.cpu().numpy()
    )

In [17]:
train_error, train_latent = extract_features(model, X_train_norm)
test_error, test_latent   = extract_features(model, X_test_norm)

In [18]:
del df
del X_train_normal, X_val_normal
del X_train_normal_tensor, X_val_normal_tensor
del X_val_tensor

In [19]:
import torch
torch.cuda.empty_cache()

In [20]:
import gc
gc.collect()

0

In [21]:
X_train_hybrid = np.hstack([train_error.reshape(-1,1), train_latent])
X_test_hybrid  = np.hstack([test_error.reshape(-1,1), test_latent])

In [24]:
import xgboost as xgb
xgclf=xgb.XGBRFClassifier(device='cuda',
                      n_estimators=100,
                      verbosity=3)
xgclf.fit(X_train_hybrid, y_train)


[19:50:42] ======== Monitor (0): HostSketchContainer ========
[19:50:42] AllReduce: 0.112144s, 1 calls @ 112144us

[19:50:42] MakeCuts: 0.112483s, 1 calls @ 112483us

[19:50:49] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (1809840, 33, 59724720).
[19:50:49] DEBUG: /__w/xgboost/xgboost/src/gbm/gbtree.cc:127: Using tree method: 0
[19:50:49] DEBUG: /__w/xgboost/xgboost/src/tree/updater_gpu_hist.cu:764: [GPU Hist]: Configure
[19:50:49] INFO: /__w/xgboost/xgboost/src/data/ellpack_page.cu:170: Ellpack is dense.
[19:50:49] ======== Monitor (0): ellpack_page ========
[19:50:49] CopyGHistToEllpack: 0.01924s, 1 calls @ 19240us

[19:50:49] InitCompressedData: 0.000325s, 1 calls @ 325us

[19:50:49] INFO: /__w/xgboost/xgboost/src/common/cuda_dr_utils.cc:180: Driver version: `580.82`
[19:50:51] ======== Monitor (0): GBTree ========
[19:50:51] BoostNewTrees: 2.03599s, 1 calls @ 2035991us

[19:50:51] CommitModel: 4e-06s, 1 calls @ 4us

[19

XGBRFClassifier(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bytree=None, device='cuda',
                early_stopping_rounds=None, enable_categorical=False,
                eval_metric=None, feature_types=None, feature_weights=None,
                gamma=None, grow_policy=None, importance_type=None,
                interaction_constraints=None, max_bin=None,
                max_cat_threshold=None, max_cat_to_onehot=None,
                max_delta_step=None, max_depth=None, max_leaves=None,
                min_child_weight=None, missing=nan, monotone_constraints=None,
                multi_strategy=None, n_estimators=100, n_jobs=None,
                num_parallel_tree=None, objective='binary:logistic',
                random_state=None, ...)

In [25]:
y_pred_xg=xgclf.predict(X_test_hybrid)

[19:50:59] DEBUG: /__w/xgboost/xgboost/src/gbm/gbtree.cc:127: Using tree method: 0
[19:50:59] DEBUG: /__w/xgboost/xgboost/src/tree/updater_gpu_hist.cu:764: [GPU Hist]: Configure


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:50:59] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [26]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("ROC-AUC:", roc_auc_score(y_test, xgclf.predict_proba(X_test_hybrid)[:,1]))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xg))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_xg))

ROC-AUC: 0.9882571393815753

Confusion Matrix:
 [[448002   6263]
 [ 14841  96470]]

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.99      0.98    454265
           1       0.94      0.87      0.90    111311

    accuracy                           0.96    565576
   macro avg       0.95      0.93      0.94    565576
weighted avg       0.96      0.96      0.96    565576



In [27]:
import torch

# Save weights
torch.save(model.state_dict(), "autoencoder_fin.pth")
import joblib

joblib.dump(xgclf, "xgbrf_model.pkl")
#joblib.dump(scaler, "scaler.pkl")
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import json

metrics = {
    "roc_auc": float(roc_auc_score(y_test,xgclf.predict_proba(X_test_hybrid)[:,1] )),
    "confusion_matrix": confusion_matrix(y_test, y_pred_xg).tolist(),
    "classification_report": classification_report(y_test, y_pred_xg, output_dict=True)
}

with open("metrics_true.json", "w") as f:
    json.dump(metrics, f, indent=4)


In [28]:
from google.colab import files

files.download("autoencoder_fin.pth")
files.download("xgbrf_model.pkl")
#files.download("scaler.pkl")
files.download("metrics_true.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>